# FactIndic prompt comparison pilot — refreshed Colab notebook

> **PAUSE — prior scope only.** This notebook consumes the old Hindi, synthetic p100, and Devanagari-based MUTANT files. It does not test Roman-script Hinglish (Hindi written in Latin letters). Do not use it for the corrected target evaluation. Wait for the GupShup h2e access and corrected Colab notebook. See `docs/dataset_scope_correction_2026-11-13.md`.

**Purpose:** compare standard and factuality-aware prompts using the same Qwen3-4B model, the same decoding settings, and a reproducible sample. The pilot contains 30 paired Hindi/synthetic-p100 article IDs plus 30 authentic MUTANT examples (180 generations total).

**Important:** this notebook does not score factuality. It creates outputs for human review. The final cell always downloads a ZIP, but it is clearly marked either `annotation_candidate` or `diagnostic_only`. A candidate still needs a human spot check before anyone annotates it.

## Before running

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
2. Run cells from top to bottom in this notebook. This notebook is self-contained; don’t mix cells from an older version.
3. When prompted, upload these three local project files. Colab accepts duplicate-download suffixes such as `(1)` automatically:
   - `hindi_baseline.jsonl` — copy of `results/baseline_hindi_test_n500_seed42/predictions.jsonl`
   - `synthetic_p100.jsonl` — copy of `results/baseline_codemixed_hindi_test_n500_seed42_p100/predictions.jsonl`
   - `mutant_authentic.jsonl` — copy of `mutant_authentic_predictions.jsonl`

The run uses greedy decoding. It does not require a Hugging Face token for this public model, though downloads may be slower without one.


In [ ]:
# Keep Hub below 2.0 for Colab's datasets package and at least 1.32 for Diffusers.
%pip -q install "transformers==5.21.0" "huggingface-hub>=1.32,<2.0" "accelerate>=1.2,<2" sentencepiece safetensors


In [ ]:
# Upload the three JSONL inputs. Duplicate names such as file (1).jsonl are normalized.
from google.colab import files
from pathlib import Path
import csv, hashlib, json, random, re, shutil, time, zipfile
from datetime import datetime, timezone

uploaded = files.upload()
REQUIRED_FILES = {"hindi_baseline.jsonl", "synthetic_p100.jsonl", "mutant_authentic.jsonl"}
resolved = {}
for uploaded_name in uploaded:
    canonical = re.sub(r" \(\d+\)(?=\.jsonl$)", "", uploaded_name)
    if canonical in REQUIRED_FILES and canonical not in resolved:
        resolved[canonical] = uploaded_name
missing = REQUIRED_FILES - set(resolved)
if missing:
    raise ValueError(f"Missing required input file(s): {sorted(missing)}")

WORK = Path("/content/factindic_prompt_pilot")
WORK.mkdir(parents=True, exist_ok=True)
for canonical, uploaded_name in resolved.items():
    target = WORK / canonical
    shutil.copy2(uploaded_name, target)
print("Inputs copied into", WORK)
print("Uploaded:", sorted(resolved))


In [ ]:
# Configuration and input validation.
import torch
import transformers, huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B"
SEED = 42
N_PAIRED = 30
N_AUTHENTIC = 30
MAX_SOURCE_TOKENS = 512
MAX_NEW_TOKENS = 256

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Colab, choose Runtime → Change runtime type → GPU, then rerun from the top.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

def read_jsonl(path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path.name}, line {line_number}: {exc}") from exc
            rows.append(row)
    return rows

hindi = read_jsonl(WORK / "hindi_baseline.jsonl")
mixed = read_jsonl(WORK / "synthetic_p100.jsonl")
auth = read_jsonl(WORK / "mutant_authentic.jsonl")
for label, rows in [("Hindi", hindi), ("synthetic p100", mixed), ("MUTANT", auth)]:
    if not rows:
        raise ValueError(f"{label} input is empty")
    if any("id" not in r or "text" not in r for r in rows):
        raise ValueError(f"{label} input must have `id` and `text` in every row")
    ids = [str(r["id"]) for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate IDs found in {label} input")
print(f"Loaded {len(hindi)} Hindi, {len(mixed)} synthetic p100, and {len(auth)} MUTANT rows")


In [ ]:
# Reproducibly choose matched Hindi/p100 IDs and balance the authentic sample by subset.
h_by_id = {str(r["id"]): r for r in hindi}
m_by_id = {str(r["id"]): r for r in mixed}
paired_ids = sorted(set(h_by_id) & set(m_by_id))
if len(paired_ids) < N_PAIRED:
    raise ValueError(f"Only {len(paired_ids)} paired Hindi/p100 IDs are available; need {N_PAIRED}.")

rng = random.Random(SEED)
selected_ids = sorted(rng.sample(paired_ids, N_PAIRED))
by_subset = {}
for row in auth:
    by_subset.setdefault(str(row.get("subset", "unknown")), []).append(row)
for subset_rows in by_subset.values():
    subset_rows.sort(key=lambda r: str(r["id"]))
subsets = sorted(by_subset)
auth_selected = []
while len(auth_selected) < min(N_AUTHENTIC, len(auth)):
    added = False
    for subset in subsets:
        if by_subset[subset] and len(auth_selected) < min(N_AUTHENTIC, len(auth)):
            auth_selected.append(by_subset[subset].pop(rng.randrange(len(by_subset[subset]))))
            added = True
    if not added:
        break
if len(auth_selected) < N_AUTHENTIC:
    raise ValueError(f"Only {len(auth_selected)} authentic rows available; need {N_AUTHENTIC}.")

examples = []
for article_id in selected_ids:
    for condition, row in [("hindi", h_by_id[article_id]), ("synthetic_p100", m_by_id[article_id])]:
        examples.append({
            "condition": condition,
            "id": article_id,
            "subset": "",
            "source": str(row["text"]),
            "baseline_prediction": str(row.get("prediction", "")),
            "baseline_truncated": row.get("truncated", ""),
        })
for row in auth_selected:
    examples.append({
        "condition": "authentic_mutant",
        "id": str(row["id"]),
        "subset": str(row.get("subset", "unknown")),
        "source": str(row["text"]),
        "baseline_prediction": str(row.get("prediction", "")),
        "baseline_truncated": row.get("truncated", ""),
    })

counts = {condition: sum(x["condition"] == condition for x in examples) for condition in ["hindi", "synthetic_p100", "authentic_mutant"]}
if counts != {"hindi": N_PAIRED, "synthetic_p100": N_PAIRED, "authentic_mutant": N_AUTHENTIC}:
    raise RuntimeError(f"Unexpected sample counts: {counts}")
print("Selected examples:", counts, "| prompt generations expected:", len(examples) * 2)
print("Authentic subset counts:", {s: sum(r.get("subset") == s for r in auth_selected) for s in subsets})


In [ ]:
# Load Qwen3-4B once on the Colab GPU.
torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
).eval()
print("Loaded", MODEL_ID, "on", model.device)


In [ ]:
# Generate both prompt arms with the same source truncation and decoding settings.
PROMPTS = {
    "standard": (
        "Summarize the source in clear, concise Hindi in 2-3 sentences (about 25-35 words). "
        "Preserve the main event and key details. Return only the summary."
    ),
    "factuality_aware": (
        "Summarize the source in clear, concise Hindi in 2-3 sentences (about 25-35 words). "
        "Use only information stated in the source. Preserve who did what to whom, and copy "
        "names, numbers, dates, negation, and attribution carefully. Do not infer or add details. "
        "If a detail is unclear, omit it. Return only the summary."
    ),
}

def trim_source(source):
    ids = tok(source, add_special_tokens=False)["input_ids"]
    visible = tok.decode(ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
    return visible, len(ids)

def generate_one(prompt_text, visible_source):
    user_text = f"{prompt_text}\n\nSource:\n{visible_source}"
    messages = [{"role": "user", "content": user_text}]
    rendered = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = tok(rendered, return_tensors="pt")
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    prompt_len = encoded["input_ids"].shape[1]
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    new_ids = generated[0, prompt_len:]
    text = tok.decode(new_ids, skip_special_tokens=True).strip()
    n_tokens = int(new_ids.shape[0])
    return text, n_tokens, n_tokens >= MAX_NEW_TOKENS

rows = []
started = time.time()
for example_no, ex in enumerate(examples, 1):
    visible_source, original_source_tokens = trim_source(ex["source"])
    for arm, prompt_text in PROMPTS.items():
        prediction, generated_tokens, cap_hit = generate_one(prompt_text, visible_source)
        rows.append({
            **ex,
            "arm": arm,
            "source_visible": visible_source,
            "source_tokens_model_tokenizer": original_source_tokens,
            "source_was_capped": original_source_tokens > MAX_SOURCE_TOKENS,
            "prediction": prediction,
            "generated_tokens": generated_tokens,
            "hit_new_token_cap": cap_hit,
        })
    # Write a local progress checkpoint after each 10 sources; this does not replace the final ZIP.
    if example_no % 10 == 0 or example_no == len(examples):
        checkpoint = WORK / "generation_checkpoint.jsonl"
        with checkpoint.open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        elapsed = time.time() - started
        print(f"{example_no}/{len(examples)} sources | {len(rows)}/180 summaries | {elapsed/60:.1f} min")
print("Generation finished in", round(time.time() - started, 1), "seconds")


In [ ]:
# Quality audit and export. This cell always downloads a ZIP, marked candidate or diagnostic-only.
if not rows:
    raise RuntimeError("No generations in memory. Run the generation cell first.")
required = {"generated_tokens", "hit_new_token_cap", "prediction", "condition", "id", "arm", "source_visible"}
missing_fields = required - set(rows[0])
if missing_fields:
    raise RuntimeError(f"Stale/incomplete generation cell: missing fields {sorted(missing_fields)}. Run this notebook from the top.")

terminal_marks = ("।", ".", "!", "?", "…", "”", '"')
cap_hits = sum(bool(r["hit_new_token_cap"]) for r in rows)
replacement_rows = sum("�" in r["prediction"] for r in rows)
empty_rows = sum(not r["prediction"].strip() for r in rows)
without_terminal = sum(not r["prediction"].rstrip().endswith(terminal_marks) for r in rows)
status = "diagnostic_only" if (cap_hits or replacement_rows or empty_rows) else "annotation_candidate"
print(f"Rows: {len(rows)} | cap hits: {cap_hits} | replacement chars: {replacement_rows} | empty: {empty_rows} | no terminal punctuation: {without_terminal}")
if status == "diagnostic_only":
    print("QUALITY GATE: diagnostic-only. Do not annotate these summaries; the ZIP will be labeled accordingly.")
else:
    print("QUALITY GATE: mechanical checks passed. Manually inspect several summaries before annotation.")
if without_terminal:
    print("Manual review signal: inspect summaries without terminal punctuation for completeness.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = WORK / f"{status}_{stamp}"
run_dir.mkdir(parents=True, exist_ok=True)
fields = list(rows[0].keys())
with (run_dir / "generations.csv").open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

if status == "annotation_candidate":
    review_fields = ["condition", "id", "subset", "arm", "source_visible", "prediction", "human_factuality", "error_type", "evidence_or_error_span", "annotator_id", "notes"]
    with (run_dir / "human_review.csv").open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=review_fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in review_fields})

input_hashes = {}
for name in sorted(REQUIRED_FILES):
    input_hashes[name] = hashlib.sha256((WORK / name).read_bytes()).hexdigest()
metadata = {
    "status": status,
    "model": MODEL_ID,
    "seed": SEED,
    "n_paired_hindi_synthetic": len(selected_ids),
    "n_authentic": len(auth_selected),
    "condition_counts": counts,
    "authentic_subset_counts": {s: sum(r.get("subset") == s for r in auth_selected) for s in subsets},
    "prompt_arms": PROMPTS,
    "max_source_tokens": MAX_SOURCE_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation": "greedy (do_sample=False)",
    "elapsed_seconds": round(time.time() - started, 2),
    "generated_token_cap_hits": cap_hits,
    "replacement_char_rows": replacement_rows,
    "empty_generation_rows": empty_rows,
    "without_terminal_punctuation_rows": without_terminal,
    "input_sha256": input_hashes,
    "selected_paired_ids": selected_ids,
    "selected_authentic_ids": [str(r["id"]) for r in auth_selected],
    "human_annotation_status": "not_started",
    "automatic_quality_checks_are_not_factuality_labels": True,
}
(run_dir / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
zip_path = WORK / f"factindic_pilot_{status}_{stamp}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(run_dir.iterdir()):
        archive.write(file_path, arcname=f"{run_dir.name}/{file_path.name}")
print("Saved", zip_path)
files.download(str(zip_path))


## How to use the download

- If named `factindic_pilot_annotation_candidate_...zip`, first inspect several rows for complete, coherent Hindi. Then human annotation can begin using `docs/factuality_annotation_protocol.md`.
- If named `factindic_pilot_diagnostic_only_...zip`, do not annotate it. Send the ZIP for diagnosis; its metadata and generations are retained for debugging.
- Keep every run ZIP. Do not overwrite earlier results. This notebook records model settings, sample IDs, input hashes, cap hits, and malformed/empty output counts.

The pilot remains small and exploratory. Report the condition and prompt-arm counts, truncation, and human annotation procedure. Do not present automatic checks or NLI as gold factuality labels.
